# Advanced RAG Techniques
Basic RAG fails when questions are ambiguous or complex. Here we will implement a Multi-Query Retriever to improve accuracy.


In [ ]:
# 1. Imports
from langchain_community.document_loaders import TextLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_openai import ChatOpenAI
from langchain.retrievers.multi_query import MultiQueryRetriever
import logging


In [ ]:
# 2. Setup Vector Store (same as basic)
with open("dummy_data2.txt", "w") as f:
    f.write("Project Alpha aims to reduce server latency by 50%. It relies on Kubernetes.")

loader = TextLoader("dummy_data2.txt")
chunks = RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=20).split_documents(loader.load())
vectorstore = Chroma.from_documents(chunks, OpenAIEmbeddings())


In [ ]:
# 3. Multi-Query Retriever
# This uses an LLM to rewrite the user's question in multiple ways to ensure we don't miss anything.
llm = ChatOpenAI(temperature=0)

retriever_from_llm = MultiQueryRetriever.from_llm(
    retriever=vectorstore.as_retriever(), llm=llm
)

# Set up logging to see the queries generated by the LLM
logging.basicConfig()
logging.getLogger("langchain.retrievers.multi_query").setLevel(logging.INFO)


In [ ]:
# 4. Test Retrieval
# The LLM will generate multiple variations of this question and search for all of them!
# unique_docs = retriever_from_llm.get_relevant_documents("Tell me about the goals of the Alpha project and its tech stack.")
# print(unique_docs)
